# Anomaly Detection for AML — Unsupervised Scoring

This notebook demonstrates unsupervised anomaly detection on transaction data
using the `aml-analytics` toolkit.

Two methods are applied and combined into an ensemble score:
- **Isolation Forest** — isolates anomalous accounts by random partitioning
- **Local Outlier Factor (LOF)** — flags accounts behaving unlike their peers

No labeled training data is required — critical in AML where confirmed cases are scarce.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from aml.synthetic import generate_transactions
from aml.anomaly import IsolationScorer, LOFScorer, ensemble_score

print("Modules loaded")

## 1. Generate data and run Isolation Forest

In [ ]:
df = generate_transactions(n=1000, seed=42)
print(f"Transactions: {len(df):,} | Accounts: {df['sender_id'].nunique()}")

iso_results = IsolationScorer(contamination=0.05).fit_score(df)
print(f"\nAccounts scored: {len(iso_results)}")
print(f"Flagged as anomalous: {iso_results['is_anomaly'].sum()}")
iso_results.head(10)

## 2. Run Local Outlier Factor

In [ ]:
lof_results = LOFScorer(contamination=0.05).fit_score(df)
print(f"Flagged by LOF: {lof_results['is_anomaly'].sum()}")
lof_results.head(10)

## 3. Ensemble score — combine both models
Accounts flagged by both models are higher-confidence alerts.

In [ ]:
ensemble = ensemble_score(df, contamination=0.05)
print(f"Flagged by Isolation Forest only : {(ensemble['iso_anomaly'] & ~ensemble['lof_anomaly']).sum() if 'iso_anomaly' in ensemble.columns else 'N/A'}")
print(f"Flagged by both models           : {ensemble['flagged_by_both'].sum()}")
print(f"Total flagged                    : {ensemble['is_anomaly'].sum()}")
ensemble.head(10)

## 4. Visualize anomaly scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Isolation Forest scores
axes[0].hist(iso_results['anomaly_score'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(iso_results[iso_results['is_anomaly']]['anomaly_score'].min(),
                color='crimson', linestyle='--', label='Anomaly threshold')
axes[0].set_title('Isolation Forest — Score Distribution', fontsize=12)
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Number of Accounts')
axes[0].legend()

# LOF scores
axes[1].hist(lof_results['anomaly_score'], bins=20, color='darkorange', edgecolor='white')
axes[1].axvline(lof_results[lof_results['is_anomaly']]['anomaly_score'].min(),
                color='crimson', linestyle='--', label='Anomaly threshold')
axes[1].set_title('LOF — Score Distribution', fontsize=12)
axes[1].set_xlabel('Anomaly Score')
axes[1].set_ylabel('Number of Accounts')
axes[1].legend()

plt.suptitle('Anomaly Score Distributions — AML Detection', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('anomaly_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Isolation Forest vs LOF scores per account
plt.figure(figsize=(9, 6))
colors = ensemble['flagged_by_both'].map({True: 'crimson', False: 'steelblue'})
plt.scatter(ensemble['iso_score'], ensemble['lof_score'],
            c=colors, alpha=0.7, edgecolors='white', linewidths=0.5, s=80)
plt.xlabel('Isolation Forest Score', fontsize=11)
plt.ylabel('LOF Score', fontsize=11)
plt.title('Ensemble Anomaly Scoring — ISO vs LOF\nRed = Flagged by Both Models', fontsize=12)
plt.tight_layout()
plt.savefig('ensemble_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature profile of top anomalous accounts
Understanding *why* an account scored high is critical for compliance teams
presenting SAR narratives.

In [ ]:
# Merge ensemble flags back with iso features for profiling
top_anomalies = iso_results[iso_results['is_anomaly']].head(5)
print("Top 5 anomalous accounts — feature profile:\n")
profile_cols = ['account_id', 'anomaly_score', 'txn_count',
                'total_amount', 'cash_ratio', 'night_ratio', 'unique_receivers']
print(top_anomalies[profile_cols].to_string(index=False))

## Summary

| Model | Accounts Flagged |
|---|---|
| Isolation Forest | See output above |
| LOF | See output above |
| Flagged by both (high confidence) | See output above |

The ensemble approach reduces false positives by requiring agreement between two
independent models — a key challenge in AML compliance where alert fatigue is a
major operational problem.

Toolkit: [github.com/Bhavesh0205/aml-analytics](https://github.com/Bhavesh0205/aml-analytics)